# Assignment 3 — Deep Learning (NNDL)

> Spec PDF: `/Users/tahamajs/Documents/uni/LLM/Deep_UT/This_year/CA3/description/NNDL_Assignment3.pdf`

This notebook contains complete, runnable code for medical image segmentation using U-Net architecture on IBSR brain segmentation dataset.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import math
import random
import time
from pathlib import Path
from glob import glob

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torch.nn.functional as F

import sys
sys.path.append('../dataset')
from Q1_dataprep import IBSRPatchDataset, get_slice_data, pad_slice

def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

seed_everything(42)

device = torch.device('cpu')
if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')

print(f"Using device: {device}")

# Question 1 — Medical Image Segmentation with U-Net

## 1-1. Dataset Preparation and Loading

In [ ]:
# Configuration
CONFIG = {
    'data_dir': '../dataset',  # Update this path to your dataset location
    'batch_size': 16,
    'epochs': 50,
    'lr': 1e-4,
    'weight_decay': 1e-5,
    'num_workers': 2,
    'seed': 42,
    'save_dir': './checkpoints',
    'num_classes': 2,  # Background and foreground (brain tissue)
    'patch_size': 128,
    'in_channels': 1
}

Path(CONFIG['save_dir']).mkdir(parents=True, exist_ok=True)

# Update these paths to point to your IBSR dataset
# Example structure:
# dataset/
#   volumes/
#     volume1.nii.gz
#     volume2.nii.gz
#   segmentations/
#     segmentation1.nii.gz
#     segmentation2.nii.gz

# You'll need to provide the actual paths to your volume and segmentation files
# volume_files = sorted(glob(f"{CONFIG['data_dir']}/volumes/*.nii.gz"))
# segmentation_files = sorted(glob(f"{CONFIG['data_dir']}/segmentations/*.nii.gz"))

# For now, we'll create a placeholder - replace with actual paths
volume_files = []  # Add your volume file paths here
segmentation_files = []  # Add your segmentation file paths here

print(f"Found {len(volume_files)} volume files")
print(f"Found {len(segmentation_files)} segmentation files")

In [ ]:
# Split dataset into train/val/test
if len(volume_files) > 0:
    # Assuming we have multiple subjects
    indices = list(range(len(volume_files)))
    random.shuffle(indices)
    
    train_size = int(0.7 * len(indices))
    val_size = int(0.15 * len(indices))
    
    train_indices = indices[:train_size]
    val_indices = indices[train_size:train_size + val_size]
    test_indices = indices[train_size + val_size:]
    
    train_volumes = [volume_files[i] for i in train_indices]
    train_segs = [segmentation_files[i] for i in train_indices]
    
    val_volumes = [volume_files[i] for i in val_indices]
    val_segs = [segmentation_files[i] for i in val_indices]
    
    test_volumes = [volume_files[i] for i in test_indices]
    test_segs = [segmentation_files[i] for i in test_indices]
    
    # Create datasets
    train_dataset = IBSRPatchDataset(train_volumes, train_segs)
    val_dataset = IBSRPatchDataset(val_volumes, val_segs)
    test_dataset = IBSRPatchDataset(test_volumes, test_segs)
    
    # Create data loaders
    train_loader = DataLoader(
        train_dataset, 
        batch_size=CONFIG['batch_size'], 
        shuffle=True, 
        num_workers=CONFIG['num_workers'],
        pin_memory=True if device.type == 'cuda' else False
    )
    
    val_loader = DataLoader(
        val_dataset, 
        batch_size=CONFIG['batch_size'], 
        shuffle=False, 
        num_workers=CONFIG['num_workers'],
        pin_memory=True if device.type == 'cuda' else False
    )
    
    test_loader = DataLoader(
        test_dataset, 
        batch_size=CONFIG['batch_size'], 
        shuffle=False, 
        num_workers=CONFIG['num_workers'],
        pin_memory=True if device.type == 'cuda' else False
    )
    
    print(f"Train samples: {len(train_dataset)}")
    print(f"Val samples: {len(val_dataset)}")
    print(f"Test samples: {len(test_dataset)}")
else:
    print("Please update volume_files and segmentation_files with actual paths to your dataset")
    # Create dummy loaders for visualization code to work
    train_loader = None
    val_loader = None
    test_loader = None

### 1-1-1. Dataset Visualization - Sample Patches

In [ ]:
# Visualize sample patches from the dataset
if train_loader is not None:
    # Get a batch of data
    sample_batch = next(iter(train_loader))
    images, masks = sample_batch
    
    num_samples = min(8, images.size(0))
    fig, axes = plt.subplots(2, num_samples, figsize=(3 * num_samples, 6))
    
    for i in range(num_samples):
        img = images[i, 0].numpy()
        mask = masks[i].numpy()
        
        # Original image
        axes[0, i].imshow(img, cmap='gray')
        axes[0, i].set_title(f'Sample {i+1}\nInput Image')
        axes[0, i].axis('off')
        
        # Ground truth mask
        axes[1, i].imshow(mask, cmap='jet', vmin=0, vmax=CONFIG['num_classes']-1)
        axes[1, i].set_title(f'Ground Truth\nMask')
        axes[1, i].axis('off')
    
    plt.tight_layout()
    plt.savefig(f"{CONFIG['save_dir']}/dataset_samples.png", dpi=300, bbox_inches='tight')
    plt.show()
    
    # Statistics visualization
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # Image intensity distribution
    all_images = []
    for batch in train_loader:
        all_images.extend(batch[0][:, 0].numpy().flatten())
        if len(all_images) > 100000:  # Limit for memory
            break
    
    axes[0].hist(all_images[:100000], bins=100, alpha=0.7, color='blue', edgecolor='black')
    axes[0].set_xlabel('Pixel Intensity')
    axes[0].set_ylabel('Frequency')
    axes[0].set_title('Image Intensity Distribution')
    axes[0].grid(True, alpha=0.3)
    
    # Mask class distribution
    all_masks = []
    for batch in train_loader:
        all_masks.extend(batch[1].numpy().flatten())
        if len(all_masks) > 100000:
            break
    
    unique, counts = np.unique(all_masks[:100000], return_counts=True)
    axes[1].bar(unique, counts, alpha=0.7, color='green', edgecolor='black')
    axes[1].set_xlabel('Class Label')
    axes[1].set_ylabel('Pixel Count')
    axes[1].set_title('Class Distribution in Masks')
    axes[1].grid(True, alpha=0.3, axis='y')
    
    # Dataset split visualization
    if len(volume_files) > 0:
        split_sizes = [len(train_dataset), len(val_dataset), len(test_dataset)]
        split_labels = ['Train', 'Validation', 'Test']
        colors = ['#3498db', '#e74c3c', '#2ecc71']
        axes[2].pie(split_sizes, labels=split_labels, autopct='%1.1f%%', 
                   colors=colors, startangle=90, textprops={'fontsize': 12})
        axes[2].set_title('Dataset Split Distribution')
    
    plt.tight_layout()
    plt.savefig(f"{CONFIG['save_dir']}/dataset_statistics.png", dpi=300, bbox_inches='tight')
    plt.show()
else:
    print("Skipping dataset visualization - please provide dataset paths")

### 1-2-1. Model Architecture Visualization

In [ ]:
# Visualize model architecture and parameters
try:
    from torchsummary import summary
    summary(model, (CONFIG['in_channels'], CONFIG['patch_size'], CONFIG['patch_size']))
except ImportError:
    print("torchsummary not available, using manual summary")
    print(f"Model parameters: {count_parameters(model):,}")
    
    # Create a visual representation of model layers
    def print_model_structure(model, prefix=""):
        for name, module in model.named_children():
            if len(list(module.children())) > 0:
                print(f"{prefix}{name}:")
                print_model_structure(module, prefix + "  ")
            else:
                params = sum(p.numel() for p in module.parameters())
                print(f"{prefix}{name}: {type(module).__name__} ({params:,} params)")
    
    print("\nModel Structure:")
    print_model_structure(model)

# Visualize model size and complexity
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Parameter distribution by layer type
layer_types = {}
layer_params = {}
for name, param in model.named_parameters():
    layer_type = name.split('.')[0]
    if layer_type not in layer_types:
        layer_types[layer_type] = 0
        layer_params[layer_type] = 0
    layer_types[layer_type] += 1
    layer_params[layer_type] += param.numel()

axes[0].bar(range(len(layer_params)), list(layer_params.values()), 
            color='steelblue', edgecolor='black', alpha=0.7)
axes[0].set_xticks(range(len(layer_params)))
axes[0].set_xticklabels(list(layer_params.keys()), rotation=45, ha='right')
axes[0].set_ylabel('Number of Parameters')
axes[0].set_title('Parameters per Layer Type')
axes[0].grid(True, alpha=0.3, axis='y')

# Model complexity pie chart
total_params = sum(layer_params.values())
axes[1].pie(layer_params.values(), labels=layer_params.keys(), autopct='%1.1f%%',
           startangle=90, textprops={'fontsize': 10})
axes[1].set_title(f'Parameter Distribution\n(Total: {total_params:,})')

plt.tight_layout()
plt.savefig(f"{CONFIG['save_dir']}/model_architecture.png", dpi=300, bbox_inches='tight')
plt.show()

## 1-8. Confusion Matrix and Detailed Analysis

In [ ]:
if len(volume_files) > 0:
    from sklearn.metrics import confusion_matrix, classification_report
    
    # Collect all predictions and targets
    model.eval()
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for images, masks in test_loader:
            images = images.to(device)
            masks = masks.to(device)
            
            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)
            
            all_preds.extend(preds.cpu().numpy().flatten())
            all_targets.extend(masks.cpu().numpy().flatten())
    
    # Create confusion matrix
    cm = confusion_matrix(all_targets, all_preds)
    
    # Plot confusion matrix
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Background', 'Foreground'],
                yticklabels=['Background', 'Foreground'])
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.title('Confusion Matrix - Test Set')
    plt.tight_layout()
    plt.savefig(f"{CONFIG['save_dir']}/confusion_matrix.png", dpi=300, bbox_inches='tight')
    plt.show()
    
    # Print classification report
    print("\nClassification Report:")
    print(classification_report(all_targets, all_preds, 
                                target_names=['Background', 'Foreground']))
    
    # Calculate additional metrics
    tn, fp, fn, tp = cm.ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    f1_score = 2 * (precision * sensitivity) / (precision + sensitivity) if (precision + sensitivity) > 0 else 0
    
    print(f"\nDetailed Metrics:")
    print(f"Sensitivity (Recall): {sensitivity:.4f}")
    print(f"Specificity: {specificity:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"F1-Score: {f1_score:.4f}")
else:
    print("Skipping confusion matrix - please provide dataset paths")

# Question 2 — Advanced Segmentation with Attention U-Net

> This question implements an enhanced U-Net architecture with attention mechanisms for improved medical image segmentation performance.

## 2-1. Configuration and Dataset Setup

In [ ]:
# Q2 Configuration
CONFIG_Q2 = {
    'data_dir': '../dataset',
    'batch_size': 16,
    'epochs': 50,
    'lr': 1e-4,
    'weight_decay': 1e-5,
    'num_workers': 2,
    'seed': 42,
    'save_dir': './checkpoints_q2',
    'num_classes': 2,
    'patch_size': 128,
    'in_channels': 1,
    'use_attention': True
}

Path(CONFIG_Q2['save_dir']).mkdir(parents=True, exist_ok=True)

# Reuse the same dataset files from Q1
# If you have different data for Q2, update these paths
volume_files_q2 = volume_files.copy() if len(volume_files) > 0 else []
segmentation_files_q2 = segmentation_files.copy() if len(segmentation_files) > 0 else []

print(f"Q2 Dataset: {len(volume_files_q2)} volume files, {len(segmentation_files_q2)} segmentation files")

## 2-2. Attention U-Net Architecture

In [ ]:
class AttentionGate(nn.Module):
    """Attention Gate module for Attention U-Net"""
    def __init__(self, F_g, F_l, F_int):
        super(AttentionGate, self).__init__()
        self.W_g = nn.Sequential(
            nn.Conv2d(F_g, F_int, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(F_int)
        )
        
        self.W_x = nn.Sequential(
            nn.Conv2d(F_l, F_int, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(F_int)
        )
        
        self.psi = nn.Sequential(
            nn.Conv2d(F_int, 1, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(1),
            nn.Sigmoid()
        )
        
        self.relu = nn.ReLU(inplace=True)
    
    def forward(self, g, x):
        # g: gating signal from decoder (smaller spatial size)
        # x: feature map from encoder (larger spatial size)
        g1 = self.W_g(g)
        x1 = self.W_x(x)
        
        # Upsample gating signal to match spatial size of x
        g1_up = F.interpolate(g1, size=x1.shape[2:], mode='bilinear', align_corners=False)
        
        psi = self.relu(g1_up + x1)
        psi = self.psi(psi)
        
        # Apply attention
        return x * psi


class AttentionUp(nn.Module):
    """Upsampling with attention gate"""
    def __init__(self, in_channels, out_channels, bilinear=True):
        super(AttentionUp, self).__init__()
        
        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
            self.conv = DoubleConv(in_channels, out_channels, in_channels // 2)
        else:
            self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
            self.conv = DoubleConv(in_channels, out_channels)
        
        # Attention gate
        self.attention = AttentionGate(F_g=in_channels // 2, F_l=out_channels, F_int=in_channels // 4)
    
    def forward(self, x1, x2):
        # x1: from decoder (smaller)
        # x2: from encoder (larger, skip connection)
        x1 = self.up(x1)
        
        # Handle size mismatch
        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]
        x1 = F.pad(x1, [diffX // 2, diffX - diffX // 2, diffY // 2, diffY - diffY // 2])
        
        # Apply attention
        x2_att = self.attention(g=x1, x=x2)
        
        # Concatenate
        x = torch.cat([x2_att, x1], dim=1)
        return self.conv(x)


class AttentionUNet(nn.Module):
    """Attention U-Net with attention gates in skip connections"""
    def __init__(self, n_channels, n_classes, bilinear=False):
        super(AttentionUNet, self).__init__()
        self.n_channels = n_channels
        self.n_classes = n_classes
        self.bilinear = bilinear
        
        # Encoder (same as U-Net)
        self.inc = DoubleConv(n_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        factor = 2 if bilinear else 1
        self.down4 = Down(512, 1024 // factor)
        
        # Decoder with attention gates
        self.up1 = AttentionUp(1024, 512 // factor, bilinear)
        self.up2 = AttentionUp(512, 256 // factor, bilinear)
        self.up3 = AttentionUp(256, 128 // factor, bilinear)
        self.up4 = AttentionUp(128, 64, bilinear)
        self.outc = OutConv(64, n_classes)
    
    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        logits = self.outc(x)
        return logits


# Create Attention U-Net model
model_q2 = AttentionUNet(n_channels=CONFIG_Q2['in_channels'], 
                        n_classes=CONFIG_Q2['num_classes'], 
                        bilinear=False)
model_q2 = model_q2.to(device)

print(f"Q2 Model parameters: {count_parameters(model_q2):,}")
print(f"\nQ2 Model architecture:")
print(model_q2)

## 2-3. Dataset Preparation for Q2

In [ ]:
# Visualize test set metrics
if test_results is not None:
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    
    # 1. Per-class Dice scores
    classes = [f'Class {i}' for i in range(CONFIG['num_classes'])]
    x_pos = np.arange(len(classes))
    colors = plt.cm.viridis(np.linspace(0, 1, len(classes)))
    
    bars1 = axes[0, 0].bar(x_pos, test_results['per_class_dice'], color=colors, 
                           alpha=0.8, edgecolor='black', linewidth=1.5)
    axes[0, 0].set_xlabel('Class', fontsize=12)
    axes[0, 0].set_ylabel('Dice Score', fontsize=12)
    axes[0, 0].set_title('Per-Class Dice Scores', fontsize=13, fontweight='bold')
    axes[0, 0].set_xticks(x_pos)
    axes[0, 0].set_xticklabels(classes)
    axes[0, 0].set_ylim([0, 1])
    axes[0, 0].grid(True, alpha=0.3, linestyle='--', axis='y')
    axes[0, 0].set_facecolor('#f8f9fa')
    # Add value labels
    for bar, val in zip(bars1, test_results['per_class_dice']):
        height = bar.get_height()
        axes[0, 0].text(bar.get_x() + bar.get_width()/2., height,
                       f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    # 2. Per-class IoU scores
    bars2 = axes[0, 1].bar(x_pos, test_results['per_class_iou'], color=colors, 
                           alpha=0.8, edgecolor='black', linewidth=1.5)
    axes[0, 1].set_xlabel('Class', fontsize=12)
    axes[0, 1].set_ylabel('IoU Score', fontsize=12)
    axes[0, 1].set_title('Per-Class IoU Scores', fontsize=13, fontweight='bold')
    axes[0, 1].set_xticks(x_pos)
    axes[0, 1].set_xticklabels(classes)
    axes[0, 1].set_ylim([0, 1])
    axes[0, 1].grid(True, alpha=0.3, linestyle='--', axis='y')
    axes[0, 1].set_facecolor('#f8f9fa')
    # Add value labels
    for bar, val in zip(bars2, test_results['per_class_iou']):
        height = bar.get_height()
        axes[0, 1].text(bar.get_x() + bar.get_width()/2., height,
                       f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    # 3. Overall metrics comparison
    metrics = ['Dice', 'IoU', 'Accuracy']
    values = [test_results['dice'], test_results['iou'], test_results['acc']]
    bars3 = axes[1, 0].bar(metrics, values, color=['#2ecc71', '#9b59b6', '#1abc9c'], 
                           alpha=0.8, edgecolor='black', linewidth=1.5)
    axes[1, 0].set_ylabel('Score', fontsize=12)
    axes[1, 0].set_title('Overall Test Metrics', fontsize=13, fontweight='bold')
    axes[1, 0].set_ylim([0, 1])
    axes[1, 0].grid(True, alpha=0.3, linestyle='--', axis='y')
    axes[1, 0].set_facecolor('#f8f9fa')
    # Add value labels
    for bar, val in zip(bars3, values):
        height = bar.get_height()
        axes[1, 0].text(bar.get_x() + bar.get_width()/2., height,
                       f'{val:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
    
    # 4. Dice vs IoU scatter
    axes[1, 1].scatter(test_results['per_class_dice'], test_results['per_class_iou'], 
                      s=200, c=colors, alpha=0.7, edgecolors='black', linewidth=2)
    for i, (d, iou) in enumerate(zip(test_results['per_class_dice'], test_results['per_class_iou'])):
        axes[1, 1].annotate(f'C{i}', (d, iou), fontsize=11, fontweight='bold', 
                           ha='center', va='center')
    axes[1, 1].set_xlabel('Dice Score', fontsize=12)
    axes[1, 1].set_ylabel('IoU Score', fontsize=12)
    axes[1, 1].set_title('Dice vs IoU (Per-Class)', fontsize=13, fontweight='bold')
    axes[1, 1].set_xlim([0, 1])
    axes[1, 1].set_ylim([0, 1])
    axes[1, 1].grid(True, alpha=0.3, linestyle='--')
    axes[1, 1].set_facecolor('#f8f9fa')
    # Add diagonal line
    axes[1, 1].plot([0, 1], [0, 1], 'r--', alpha=0.5, linewidth=1)
    
    plt.suptitle('Test Set Evaluation Metrics', fontsize=16, fontweight='bold', y=0.995)
    plt.tight_layout()
    plt.savefig(f"{CONFIG['save_dir']}/test_metrics.png", dpi=300, bbox_inches='tight')
    plt.show()
else:
    print("Skipping metrics visualization - test results not available")

### 1-6-2. Confusion Matrix Visualization

In [ ]:
# Create confusion matrix for pixel-level classification
if test_results is not None:
    from sklearn.metrics import confusion_matrix, classification_report
    
    # Flatten predictions and targets
    y_true = test_results['all_targets'].numpy().flatten()
    y_pred = test_results['all_preds'].numpy().flatten()
    
    # Compute confusion matrix
    cm = confusion_matrix(y_true, y_pred, labels=list(range(CONFIG['num_classes'])))
    
    # Normalize confusion matrix
    cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    
    # Plot confusion matrices
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Absolute values
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0], 
                cbar_kws={'label': 'Count'}, square=True, linewidths=0.5)
    axes[0].set_xlabel('Predicted Class', fontsize=12)
    axes[0].set_ylabel('True Class', fontsize=12)
    axes[0].set_title('Confusion Matrix (Absolute)', fontsize=13, fontweight='bold')
    axes[0].set_xticklabels([f'Class {i}' for i in range(CONFIG['num_classes'])])
    axes[0].set_yticklabels([f'Class {i}' for i in range(CONFIG['num_classes'])])
    
    # Normalized values
    sns.heatmap(cm_normalized, annot=True, fmt='.3f', cmap='Oranges', ax=axes[1],
                cbar_kws={'label': 'Proportion'}, square=True, linewidths=0.5)
    axes[1].set_xlabel('Predicted Class', fontsize=12)
    axes[1].set_ylabel('True Class', fontsize=12)
    axes[1].set_title('Confusion Matrix (Normalized)', fontsize=13, fontweight='bold')
    axes[1].set_xticklabels([f'Class {i}' for i in range(CONFIG['num_classes'])])
    axes[1].set_yticklabels([f'Class {i}' for i in range(CONFIG['num_classes'])])
    
    plt.tight_layout()
    plt.savefig(f"{CONFIG['save_dir']}/confusion_matrix.png", dpi=300, bbox_inches='tight')
    plt.show()
    
    # Print classification report
    print("\n" + "=" * 60)
    print("CLASSIFICATION REPORT")
    print("=" * 60)
    print(classification_report(y_true, y_pred, 
                                target_names=[f'Class {i}' for i in range(CONFIG['num_classes'])]))
    print("=" * 60)
else:
    print("Skipping confusion matrix - test results not available")

In [ ]:
# Split dataset for Q2 (reuse same splitting logic)
if len(volume_files_q2) > 0:
    indices_q2 = list(range(len(volume_files_q2)))
    random.shuffle(indices_q2)
    
    train_size_q2 = int(0.7 * len(indices_q2))
    val_size_q2 = int(0.15 * len(indices_q2))
    
    train_indices_q2 = indices_q2[:train_size_q2]
    val_indices_q2 = indices_q2[train_size_q2:train_size_q2 + val_size_q2]
    test_indices_q2 = indices_q2[train_size_q2 + val_size_q2:]
    
    train_volumes_q2 = [volume_files_q2[i] for i in train_indices_q2]
    train_segs_q2 = [segmentation_files_q2[i] for i in train_indices_q2]
    
    val_volumes_q2 = [volume_files_q2[i] for i in val_indices_q2]
    val_segs_q2 = [segmentation_files_q2[i] for i in val_indices_q2]
    
    test_volumes_q2 = [volume_files_q2[i] for i in test_indices_q2]
    test_segs_q2 = [segmentation_files_q2[i] for i in test_indices_q2]
    
    # Create datasets
    train_dataset_q2 = IBSRPatchDataset(train_volumes_q2, train_segs_q2)
    val_dataset_q2 = IBSRPatchDataset(val_volumes_q2, val_segs_q2)
    test_dataset_q2 = IBSRPatchDataset(test_volumes_q2, test_segs_q2)
    
    # Create data loaders
    train_loader_q2 = DataLoader(
        train_dataset_q2, 
        batch_size=CONFIG_Q2['batch_size'], 
        shuffle=True, 
        num_workers=CONFIG_Q2['num_workers'],
        pin_memory=True if device.type == 'cuda' else False
    )
    
    val_loader_q2 = DataLoader(
        val_dataset_q2, 
        batch_size=CONFIG_Q2['batch_size'], 
        shuffle=False, 
        num_workers=CONFIG_Q2['num_workers'],
        pin_memory=True if device.type == 'cuda' else False
    )
    
    test_loader_q2 = DataLoader(
        test_dataset_q2, 
        batch_size=CONFIG_Q2['batch_size'], 
        shuffle=False, 
        num_workers=CONFIG_Q2['num_workers'],
        pin_memory=True if device.type == 'cuda' else False
    )
    
    print(f"Q2 Train samples: {len(train_dataset_q2)}")
    print(f"Q2 Val samples: {len(val_dataset_q2)}")
    print(f"Q2 Test samples: {len(test_dataset_q2)}")
else:
    print("Please provide dataset files for Q2")

### 1-7-1. Error Analysis and Distribution Visualization

In [ ]:
# Error analysis visualization
if test_results is not None:
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    
    # 1. Prediction distribution
    pred_flat = test_results['all_preds'].numpy().flatten()
    target_flat = test_results['all_targets'].numpy().flatten()
    
    unique_pred, counts_pred = np.unique(pred_flat, return_counts=True)
    unique_target, counts_target = np.unique(target_flat, return_counts=True)
    
    x = np.arange(CONFIG['num_classes'])
    width = 0.35
    axes[0, 0].bar(x - width/2, [counts_target[i] if i < len(unique_target) and unique_target[i] == i else 0 
                                  for i in range(CONFIG['num_classes'])], 
                  width, label='Ground Truth', color='#3498db', alpha=0.8)
    axes[0, 0].bar(x + width/2, [counts_pred[i] if i < len(unique_pred) and unique_pred[i] == i else 0 
                                 for i in range(CONFIG['num_classes'])], 
                  width, label='Predictions', color='#e74c3c', alpha=0.8)
    axes[0, 0].set_xlabel('Class', fontsize=12)
    axes[0, 0].set_ylabel('Pixel Count', fontsize=12)
    axes[0, 0].set_title('Class Distribution: Ground Truth vs Predictions', fontsize=13, fontweight='bold')
    axes[0, 0].set_xticks(x)
    axes[0, 0].set_xticklabels([f'Class {i}' for i in range(CONFIG['num_classes'])])
    axes[0, 0].legend(fontsize=10)
    axes[0, 0].grid(True, alpha=0.3, linestyle='--', axis='y')
    axes[0, 0].set_facecolor('#f8f9fa')
    
    # 2. Error rate per class
    error_per_class = []
    for i in range(CONFIG['num_classes']):
        mask_class = (target_flat == i)
        if mask_class.sum() > 0:
            errors = (pred_flat[mask_class] != target_flat[mask_class]).sum()
            error_rate = errors / mask_class.sum()
        else:
            error_rate = 0
        error_per_class.append(error_rate)
    
    bars = axes[0, 1].bar(range(CONFIG['num_classes']), error_per_class, 
                         color=['#e74c3c' if e > 0.1 else '#2ecc71' for e in error_per_class],
                         alpha=0.8, edgecolor='black', linewidth=1.5)
    axes[0, 1].set_xlabel('Class', fontsize=12)
    axes[0, 1].set_ylabel('Error Rate', fontsize=12)
    axes[0, 1].set_title('Error Rate per Class', fontsize=13, fontweight='bold')
    axes[0, 1].set_xticks(range(CONFIG['num_classes']))
    axes[0, 1].set_xticklabels([f'Class {i}' for i in range(CONFIG['num_classes'])])
    axes[0, 1].set_ylim([0, 1])
    axes[0, 1].grid(True, alpha=0.3, linestyle='--', axis='y')
    axes[0, 1].set_facecolor('#f8f9fa')
    # Add value labels
    for bar, val in zip(bars, error_per_class):
        height = bar.get_height()
        axes[0, 1].text(bar.get_x() + bar.get_width()/2., height,
                       f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    # 3. Dice score distribution across samples
    sample_dice_scores = []
    for i in range(min(100, test_results['all_images'].size(0))):  # Limit for performance
        dice = calculate_dice_score(
            test_results['all_preds'][i:i+1],
            test_results['all_targets'][i:i+1],
            CONFIG['num_classes']
        ).mean().item()
        sample_dice_scores.append(dice)
    
    axes[1, 0].hist(sample_dice_scores, bins=30, color='#3498db', alpha=0.7, 
                   edgecolor='black', linewidth=1.5)
    axes[1, 0].axvline(np.mean(sample_dice_scores), color='red', linestyle='--', 
                      linewidth=2, label=f'Mean: {np.mean(sample_dice_scores):.3f}')
    axes[1, 0].axvline(np.median(sample_dice_scores), color='green', linestyle='--', 
                       linewidth=2, label=f'Median: {np.median(sample_dice_scores):.3f}')
    axes[1, 0].set_xlabel('Dice Score', fontsize=12)
    axes[1, 0].set_ylabel('Frequency', fontsize=12)
    axes[1, 0].set_title('Dice Score Distribution Across Samples', fontsize=13, fontweight='bold')
    axes[1, 0].legend(fontsize=10)
    axes[1, 0].grid(True, alpha=0.3, linestyle='--', axis='y')
    axes[1, 0].set_facecolor('#f8f9fa')
    
    # 4. Confidence/Probability distribution
    prob_flat = F.softmax(test_results['all_outputs'], dim=1).max(dim=1)[0].numpy().flatten()
    axes[1, 1].hist(prob_flat, bins=50, color='#9b59b6', alpha=0.7, 
                   edgecolor='black', linewidth=1.5)
    axes[1, 1].axvline(np.mean(prob_flat), color='red', linestyle='--', 
                      linewidth=2, label=f'Mean: {np.mean(prob_flat):.3f}')
    axes[1, 1].set_xlabel('Prediction Confidence', fontsize=12)
    axes[1, 1].set_ylabel('Frequency', fontsize=12)
    axes[1, 1].set_title('Prediction Confidence Distribution', fontsize=13, fontweight='bold')
    axes[1, 1].legend(fontsize=10)
    axes[1, 1].grid(True, alpha=0.3, linestyle='--', axis='y')
    axes[1, 1].set_facecolor('#f8f9fa')
    
    plt.suptitle('Error Analysis and Distribution', fontsize=16, fontweight='bold', y=0.995)
    plt.tight_layout()
    plt.savefig(f"{CONFIG['save_dir']}/error_analysis.png", dpi=300, bbox_inches='tight')
    plt.show()
else:
    print("Skipping error analysis - test results not available")

### 1-7-2. Best and Worst Predictions

In [ ]:
# Visualize best and worst predictions
if test_results is not None:
    # Calculate dice scores for all samples
    sample_dice_scores = []
    for i in range(test_results['all_images'].size(0)):
        dice = calculate_dice_score(
            test_results['all_preds'][i:i+1],
            test_results['all_targets'][i:i+1],
            CONFIG['num_classes']
        ).mean().item()
        sample_dice_scores.append(dice)
    
    sample_dice_scores = np.array(sample_dice_scores)
    
    # Get best and worst indices
    num_show = 3
    best_indices = np.argsort(sample_dice_scores)[-num_show:][::-1]
    worst_indices = np.argsort(sample_dice_scores)[:num_show]
    
    fig, axes = plt.subplots(num_show, 5, figsize=(20, 4 * num_show))
    
    for row, (indices, title_prefix) in enumerate([(best_indices, 'Best'), (worst_indices, 'Worst')]):
        for col, idx in enumerate(indices):
            img = test_results['all_images'][idx, 0].numpy()
            mask = test_results['all_targets'][idx].numpy()
            pred = test_results['all_preds'][idx].numpy()
            dice_score = sample_dice_scores[idx]
            
            # Input
            axes[row, 0].imshow(img, cmap='gray')
            if col == 0:
                axes[row, 0].set_ylabel(f'{title_prefix}\nPredictions', fontsize=12, fontweight='bold')
            axes[row, 0].set_title(f'Sample {idx+1}\nInput', fontsize=10)
            axes[row, 0].axis('off')
            
            # Ground truth
            axes[row, 1].imshow(mask, cmap='jet', vmin=0, vmax=CONFIG['num_classes']-1)
            axes[row, 1].set_title('Ground Truth', fontsize=10)
            axes[row, 1].axis('off')
            
            # Prediction
            axes[row, 2].imshow(pred, cmap='jet', vmin=0, vmax=CONFIG['num_classes']-1)
            axes[row, 2].set_title(f'Prediction\nDice: {dice_score:.3f}', fontsize=10)
            axes[row, 2].axis('off')
            
            # Overlay
            axes[row, 3].imshow(img, cmap='gray', alpha=0.6)
            overlay = pred.copy().astype(float)
            overlay[overlay == 0] = np.nan
            axes[row, 3].imshow(overlay, cmap='jet', alpha=0.4, vmin=0, vmax=CONFIG['num_classes']-1)
            axes[row, 3].set_title('Overlay', fontsize=10)
            axes[row, 3].axis('off')
            
            # Error map
            error_map = np.zeros_like(mask, dtype=float)
            error_map[(mask != pred) & (mask != 0)] = 1
            error_map[(mask != pred) & (pred != 0)] = 2
            error_map[(mask == pred) & (mask != 0)] = 0.5
            axes[row, 4].imshow(error_map, cmap='RdYlGn', vmin=0, vmax=2)
            axes[row, 4].set_title('Error Map', fontsize=10)
            axes[row, 4].axis('off')
    
    plt.suptitle('Best and Worst Predictions Comparison', fontsize=16, fontweight='bold', y=0.995)
    plt.tight_layout()
    plt.savefig(f"{CONFIG['save_dir']}/best_worst_predictions.png", dpi=300, bbox_inches='tight')
    plt.show()
else:
    print("Skipping best/worst visualization - test results not available")

## Summary and Final Report

### 1-7-3. Probability Maps and Confidence Visualization

In [ ]:
# Visualize probability maps and confidence scores
if test_results is not None:
    num_samples = min(6, test_results['all_images'].size(0))
    fig, axes = plt.subplots(num_samples, 4, figsize=(16, 4 * num_samples))
    if num_samples == 1:
        axes = axes.reshape(1, -1)
    
    for idx in range(num_samples):
        img = test_results['all_images'][idx, 0].numpy()
        mask = test_results['all_targets'][idx].numpy()
        pred = test_results['all_preds'][idx].numpy()
        output = test_results['all_outputs'][idx]
        prob = F.softmax(output, dim=0).numpy()
        
        # Get max probability (confidence) for each pixel
        confidence = prob.max(axis=0)
        predicted_class = prob.argmax(axis=0)
        
        # Input image
        axes[idx, 0].imshow(img, cmap='gray')
        axes[idx, 0].set_title(f'Sample {idx+1}\nInput Image', fontsize=11)
        axes[idx, 0].axis('off')
        
        # Ground truth
        axes[idx, 1].imshow(mask, cmap='jet', vmin=0, vmax=CONFIG['num_classes']-1)
        axes[idx, 1].set_title('Ground Truth', fontsize=11)
        axes[idx, 1].axis('off')
        
        # Prediction with confidence overlay
        im = axes[idx, 2].imshow(pred, cmap='jet', vmin=0, vmax=CONFIG['num_classes']-1, alpha=0.7)
        conf_im = axes[idx, 2].imshow(confidence, cmap='hot', alpha=0.3, vmin=0, vmax=1)
        axes[idx, 2].set_title('Prediction + Confidence', fontsize=11)
        axes[idx, 2].axis('off')
        
        # Confidence map only
        conf_map = axes[idx, 3].imshow(confidence, cmap='viridis', vmin=0, vmax=1)
        axes[idx, 3].set_title(f'Confidence Map\n(Mean: {confidence.mean():.3f})', fontsize=11)
        axes[idx, 3].axis('off')
        plt.colorbar(conf_map, ax=axes[idx, 3], fraction=0.046, pad=0.04)
    
    plt.suptitle('Probability Maps and Confidence Visualization', fontsize=16, fontweight='bold', y=0.995)
    plt.tight_layout()
    plt.savefig(f"{CONFIG['save_dir']}/probability_maps.png", dpi=300, bbox_inches='tight')
    plt.show()
    
    # Confidence distribution by class
    fig, axes = plt.subplots(1, CONFIG['num_classes'], figsize=(5 * CONFIG['num_classes'], 4))
    if CONFIG['num_classes'] == 1:
        axes = [axes]
    
    for class_idx in range(CONFIG['num_classes']):
        class_confidences = []
        for i in range(min(100, test_results['all_images'].size(0))):
            output = test_results['all_outputs'][i]
            prob = F.softmax(output, dim=0).numpy()
            target = test_results['all_targets'][i].numpy()
            # Get confidence for pixels of this class
            class_mask = (target == class_idx)
            if class_mask.sum() > 0:
                class_conf = prob[class_idx][class_mask]
                class_confidences.extend(class_conf.tolist())
        
        if len(class_confidences) > 0:
            axes[class_idx].hist(class_confidences, bins=50, color=plt.cm.viridis(class_idx / CONFIG['num_classes']), 
                               alpha=0.7, edgecolor='black')
            axes[class_idx].axvline(np.mean(class_confidences), color='red', linestyle='--', 
                                  linewidth=2, label=f'Mean: {np.mean(class_confidences):.3f}')
            axes[class_idx].set_xlabel('Confidence', fontsize=11)
            axes[class_idx].set_ylabel('Frequency', fontsize=11)
            axes[class_idx].set_title(f'Class {class_idx} Confidence Distribution', fontsize=12, fontweight='bold')
            axes[class_idx].legend(fontsize=10)
            axes[class_idx].grid(True, alpha=0.3, linestyle='--', axis='y')
            axes[class_idx].set_facecolor('#f8f9fa')
    
    plt.tight_layout()
    plt.savefig(f"{CONFIG['save_dir']}/confidence_by_class.png", dpi=300, bbox_inches='tight')
    plt.show()
else:
    print("Skipping probability visualization - test results not available")

### 1-7-4. Training Progress Animation Data (for external tools)

In [ ]:
# Save training history to JSON for external visualization tools
import json

if len(history['train_loss']) > 0:
    # Prepare data for export
    export_data = {
        'config': CONFIG,
        'history': {k: v for k, v in history.items() if isinstance(v, list)},
        'best_epoch': int(np.argmax(history['val_dice'])) if len(history['val_dice']) > 0 else 0,
        'best_val_dice': float(max(history['val_dice'])) if len(history['val_dice']) > 0 else 0.0
    }
    
    # Add test results if available
    if test_results is not None:
        export_data['test_results'] = {
            'loss': float(test_results['loss']),
            'dice': float(test_results['dice']),
            'iou': float(test_results['iou']),
            'acc': float(test_results['acc']),
            'per_class_dice': [float(x) for x in test_results['per_class_dice']],
            'per_class_iou': [float(x) for x in test_results['per_class_iou']]
        }
    
    # Save to JSON
    with open(f"{CONFIG['save_dir']}/training_history.json", 'w') as f:
        json.dump(export_data, f, indent=2)
    
    print(f"✓ Training history saved to {CONFIG['save_dir']}/training_history.json")
    
    # Create a summary CSV file
    import pandas as pd
    
    df_history = pd.DataFrame({
        'epoch': range(1, len(history['train_loss']) + 1),
        'train_loss': history['train_loss'],
        'val_loss': history['val_loss'],
        'train_dice': history['train_dice'],
        'val_dice': history['val_dice'],
        'train_iou': history['train_iou'],
        'val_iou': history['val_iou'],
        'train_acc': history['train_acc'],
        'val_acc': history['val_acc'],
        'lr': history.get('lr', [0] * len(history['train_loss']))
    })
    
    df_history.to_csv(f"{CONFIG['save_dir']}/training_history.csv", index=False)
    print(f"✓ Training history CSV saved to {CONFIG['save_dir']}/training_history.csv")
    
    # Display summary statistics
    print("\n" + "=" * 60)
    print("TRAINING SUMMARY STATISTICS")
    print("=" * 60)
    print(f"Total epochs trained: {len(history['train_loss'])}")
    print(f"Best validation Dice: {max(history['val_dice']):.4f} at epoch {np.argmax(history['val_dice']) + 1}")
    print(f"Final training Dice: {history['train_dice'][-1]:.4f}")
    print(f"Final validation Dice: {history['val_dice'][-1]:.4f}")
    print(f"Training/Validation gap (Dice): {history['train_dice'][-1] - history['val_dice'][-1]:.4f}")
    if 'lr' in history and len(history['lr']) > 0:
        print(f"Initial learning rate: {history['lr'][0]:.6f}")
        print(f"Final learning rate: {history['lr'][-1]:.6f}")
    print("=" * 60)
else:
    print("No training history available for export")

### 1-7-5. Model Comparison and Ablation Study Visualization

In [ ]:
# Create comparison visualization if you train multiple models
# This is a template for comparing different model configurations

comparison_data = {
    'U-Net (Baseline)': {
        'dice': test_results['dice'] if test_results is not None else 0.0,
        'iou': test_results['iou'] if test_results is not None else 0.0,
        'acc': test_results['acc'] if test_results is not None else 0.0,
        'params': count_parameters(model)
    }
}

# If you have other models (e.g., Attention U-Net from Q2), add them here
# comparison_data['Attention U-Net'] = {...}

if len(comparison_data) > 1:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    models = list(comparison_data.keys())
    dice_scores = [comparison_data[m]['dice'] for m in models]
    iou_scores = [comparison_data[m]['iou'] for m in models]
    acc_scores = [comparison_data[m]['acc'] for m in models]
    
    x = np.arange(len(models))
    width = 0.25
    
    axes[0].bar(x - width, dice_scores, width, label='Dice', color='#2ecc71', alpha=0.8)
    axes[0].bar(x, iou_scores, width, label='IoU', color='#9b59b6', alpha=0.8)
    axes[0].bar(x + width, acc_scores, width, label='Accuracy', color='#3498db', alpha=0.8)
    axes[0].set_xlabel('Model', fontsize=12)
    axes[0].set_ylabel('Score', fontsize=12)
    axes[0].set_title('Model Comparison: Metrics', fontsize=13, fontweight='bold')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(models, rotation=45, ha='right')
    axes[0].legend(fontsize=10)
    axes[0].grid(True, alpha=0.3, linestyle='--', axis='y')
    axes[0].set_ylim([0, 1])
    axes[0].set_facecolor('#f8f9fa')
    
    # Parameter count comparison
    param_counts = [comparison_data[m]['params'] for m in models]
    axes[1].bar(models, param_counts, color='#e74c3c', alpha=0.8, edgecolor='black')
    axes[1].set_xlabel('Model', fontsize=12)
    axes[1].set_ylabel('Number of Parameters', fontsize=12)
    axes[1].set_title('Model Comparison: Parameter Count', fontsize=13, fontweight='bold')
    axes[1].tick_params(axis='x', rotation=45)
    axes[1].grid(True, alpha=0.3, linestyle='--', axis='y')
    axes[1].set_facecolor('#f8f9fa')
    # Add value labels
    for i, (model, params) in enumerate(zip(models, param_counts)):
        axes[1].text(i, params, f'{params:,}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    
    # Efficiency plot (Score vs Parameters)
    axes[2].scatter(param_counts, dice_scores, s=200, alpha=0.7, c=range(len(models)), 
                   cmap='viridis', edgecolors='black', linewidth=2)
    for i, model in enumerate(models):
        axes[2].annotate(model, (param_counts[i], dice_scores[i]), 
                        fontsize=10, ha='center', va='bottom')
    axes[2].set_xlabel('Number of Parameters', fontsize=12)
    axes[2].set_ylabel('Dice Score', fontsize=12)
    axes[2].set_title('Model Efficiency: Score vs Parameters', fontsize=13, fontweight='bold')
    axes[2].grid(True, alpha=0.3, linestyle='--')
    axes[2].set_facecolor('#f8f9fa')
    
    plt.tight_layout()
    plt.savefig(f"{CONFIG['save_dir']}/model_comparison.png", dpi=300, bbox_inches='tight')
    plt.show()
else:
    print("Add more models to comparison_data dictionary to visualize model comparison")
    print("Current models:", list(comparison_data.keys()))

In [ ]:
# Generate final summary report
if test_results is not None and len(history['train_loss']) > 0:
    print("\n" + "=" * 80)
    print("FINAL MODEL PERFORMANCE SUMMARY")
    print("=" * 80)
    
    print(f"\n📊 TRAINING SUMMARY:")
    print(f"  • Total Epochs: {len(history['train_loss'])}")
    print(f"  • Best Epoch: {np.argmax(history['val_dice']) + 1}")
    print(f"  • Final Training Loss: {history['train_loss'][-1]:.4f}")
    print(f"  • Final Validation Loss: {history['val_loss'][-1]:.4f}")
    
    print(f"\n🎯 TEST SET PERFORMANCE:")
    print(f"  • Test Loss: {test_results['loss']:.4f}")
    print(f"  • Average Dice Score: {test_results['dice']:.4f}")
    print(f"  • Average IoU Score: {test_results['iou']:.4f}")
    print(f"  • Pixel Accuracy: {test_results['acc']:.4f}")
    
    print(f"\n📈 PER-CLASS METRICS:")
    for i in range(CONFIG['num_classes']):
        print(f"  • Class {i}:")
        print(f"    - Dice: {test_results['per_class_dice'][i]:.4f}")
        print(f"    - IoU: {test_results['per_class_iou'][i]:.4f}")
    
    print(f"\n💾 SAVED FILES:")
    print(f"  • Model checkpoint: {CONFIG['save_dir']}/best_model.pth")
    print(f"  • Training curves: {CONFIG['save_dir']}/training_curves.png")
    print(f"  • Comprehensive analysis: {CONFIG['save_dir']}/training_curves_comprehensive.png")
    print(f"  • Test metrics: {CONFIG['save_dir']}/test_metrics.png")
    print(f"  • Confusion matrix: {CONFIG['save_dir']}/confusion_matrix.png")
    print(f"  • Predictions: {CONFIG['save_dir']}/predictions.png")
    print(f"  • Comprehensive predictions: {CONFIG['save_dir']}/predictions_comprehensive.png")
    print(f"  • Error analysis: {CONFIG['save_dir']}/error_analysis.png")
    print(f"  • Best/worst predictions: {CONFIG['save_dir']}/best_worst_predictions.png")
    
    print("\n" + "=" * 80)
else:
    print("Summary not available - please run training and evaluation first")

## 2-4. Training Attention U-Net

In [ ]:
# Optimizer and scheduler for Q2
optimizer_q2 = optim.Adam(model_q2.parameters(), lr=CONFIG_Q2['lr'], weight_decay=CONFIG_Q2['weight_decay'])
scheduler_q2 = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_q2, mode='min', factor=0.5, patience=5, verbose=True
)

# Loss function (same as Q1)
criterion_q2 = CombinedLoss(dice_weight=0.5, ce_weight=0.5)

# Training history for Q2
history_q2 = {
    'train_loss': [],
    'val_loss': [],
    'train_dice': [],
    'val_dice': [],
    'train_iou': [],
    'val_iou': [],
    'train_acc': [],
    'val_acc': []
}

# Training loop for Q2
if len(volume_files_q2) > 0:
    best_val_dice_q2 = 0.0
    patience_counter_q2 = 0
    patience_q2 = 10
    
    print("Starting Q2 training (Attention U-Net)...")
    print(f"Total epochs: {CONFIG_Q2['epochs']}")
    print("-" * 60)
    
    for epoch in range(CONFIG_Q2['epochs']):
        print(f"\nEpoch {epoch + 1}/{CONFIG_Q2['epochs']}")
        print(f"Learning rate: {optimizer_q2.param_groups[0]['lr']:.6f}")
        
        # Train
        train_loss, train_dice, train_iou, train_acc = train_epoch(
            model_q2, train_loader_q2, criterion_q2, optimizer_q2, device
        )
        
        # Validate
        val_loss, val_dice, val_iou, val_acc = validate_epoch(
            model_q2, val_loader_q2, criterion_q2, device
        )
        
        # Update learning rate
        scheduler_q2.step(val_loss)
        
        # Save history
        history_q2['train_loss'].append(train_loss)
        history_q2['val_loss'].append(val_loss)
        history_q2['train_dice'].append(train_dice)
        history_q2['val_dice'].append(val_dice)
        history_q2['train_iou'].append(train_iou)
        history_q2['val_iou'].append(val_iou)
        history_q2['train_acc'].append(train_acc)
        history_q2['val_acc'].append(val_acc)
        
        print(f"\nTrain - Loss: {train_loss:.4f}, Dice: {train_dice:.4f}, "
              f"IoU: {train_iou:.4f}, Acc: {train_acc:.4f}")
        print(f"Val   - Loss: {val_loss:.4f}, Dice: {val_dice:.4f}, "
              f"IoU: {val_iou:.4f}, Acc: {val_acc:.4f}")
        
        # Save best model
        if val_dice > best_val_dice_q2:
            best_val_dice_q2 = val_dice
            patience_counter_q2 = 0
            torch.save({
                'epoch': epoch,
                'model_state_dict': model_q2.state_dict(),
                'optimizer_state_dict': optimizer_q2.state_dict(),
                'val_dice': val_dice,
                'history': history_q2
            }, f"{CONFIG_Q2['save_dir']}/best_model_q2.pth")
            print(f"✓ Saved best Q2 model (Dice: {best_val_dice_q2:.4f})")
        else:
            patience_counter_q2 += 1
            if patience_counter_q2 >= patience_q2:
                print(f"\nEarly stopping at epoch {epoch + 1}")
                break
    
    print("\n" + "=" * 60)
    print("Q2 Training completed!")
    print(f"Best validation Dice score: {best_val_dice_q2:.4f}")
else:
    print("Skipping Q2 training - please provide dataset paths")

In [ ]:
if len(history['train_loss']) > 0 and len(history_q2['train_loss']) > 0:
    # Compare training curves
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    epochs_q1 = range(1, len(history['train_loss']) + 1)
    epochs_q2 = range(1, len(history_q2['train_loss']) + 1)
    
    # Loss comparison
    axes[0, 0].plot(epochs_q1, history['val_loss'], label='U-Net Val Loss', linewidth=2, linestyle='--')
    axes[0, 0].plot(epochs_q2, history_q2['val_loss'], label='Attention U-Net Val Loss', linewidth=2)
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Validation Loss')
    axes[0, 0].set_title('Validation Loss Comparison')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # Dice comparison
    axes[0, 1].plot(epochs_q1, history['val_dice'], label='U-Net Val Dice', linewidth=2, linestyle='--')
    axes[0, 1].plot(epochs_q2, history_q2['val_dice'], label='Attention U-Net Val Dice', linewidth=2)
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Validation Dice Score')
    axes[0, 1].set_title('Validation Dice Score Comparison')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # IoU comparison
    axes[1, 0].plot(epochs_q1, history['val_iou'], label='U-Net Val IoU', linewidth=2, linestyle='--')
    axes[1, 0].plot(epochs_q2, history_q2['val_iou'], label='Attention U-Net Val IoU', linewidth=2)
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('Validation IoU Score')
    axes[1, 0].set_title('Validation IoU Score Comparison')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # Accuracy comparison
    axes[1, 1].plot(epochs_q1, history['val_acc'], label='U-Net Val Acc', linewidth=2, linestyle='--')
    axes[1, 1].plot(epochs_q2, history_q2['val_acc'], label='Attention U-Net Val Acc', linewidth=2)
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Validation Pixel Accuracy')
    axes[1, 1].set_title('Validation Accuracy Comparison')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f"{CONFIG_Q2['save_dir']}/comparison_curves.png", dpi=300, bbox_inches='tight')
    plt.show()
    
    # Print summary comparison
    print("\n" + "=" * 60)
    print("MODEL COMPARISON SUMMARY")
    print("=" * 60)
    print(f"{'Metric':<25} {'U-Net':<15} {'Attention U-Net':<15}")
    print("-" * 60)
    print(f"{'Best Val Dice':<25} {max(history['val_dice']):<15.4f} {max(history_q2['val_dice']):<15.4f}")
    print(f"{'Best Val IoU':<25} {max(history['val_iou']):<15.4f} {max(history_q2['val_iou']):<15.4f}")
    print(f"{'Best Val Accuracy':<25} {max(history['val_acc']):<15.4f} {max(history_q2['val_acc']):<15.4f}")
    print(f"{'Parameters':<25} {count_parameters(model):<15,} {count_parameters(model_q2):<15,}")
    print("=" * 60)
else:
    print("Both models need to be trained for comparison")

## 2-5. Q2 Training Curves and Evaluation

In [ ]:
if len(history_q2['train_loss']) > 0:
    # Plot training curves
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Loss curves
    axes[0, 0].plot(history_q2['train_loss'], label='Train Loss', linewidth=2)
    axes[0, 0].plot(history_q2['val_loss'], label='Val Loss', linewidth=2)
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].set_title('Q2: Training and Validation Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # Dice score curves
    axes[0, 1].plot(history_q2['train_dice'], label='Train Dice', linewidth=2)
    axes[0, 1].plot(history_q2['val_dice'], label='Val Dice', linewidth=2)
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Dice Score')
    axes[0, 1].set_title('Q2: Training and Validation Dice Score')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # IoU curves
    axes[1, 0].plot(history_q2['train_iou'], label='Train IoU', linewidth=2)
    axes[1, 0].plot(history_q2['val_iou'], label='Val IoU', linewidth=2)
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('IoU Score')
    axes[1, 0].set_title('Q2: Training and Validation IoU Score')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # Accuracy curves
    axes[1, 1].plot(history_q2['train_acc'], label='Train Acc', linewidth=2)
    axes[1, 1].plot(history_q2['val_acc'], label='Val Acc', linewidth=2)
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Pixel Accuracy')
    axes[1, 1].set_title('Q2: Training and Validation Pixel Accuracy')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f"{CONFIG_Q2['save_dir']}/training_curves_q2.png", dpi=300, bbox_inches='tight')
    plt.show()
    
    # Test set evaluation
    checkpoint_q2 = torch.load(f"{CONFIG_Q2['save_dir']}/best_model_q2.pth")
    model_q2.load_state_dict(checkpoint_q2['model_state_dict'])
    print(f"Loaded best Q2 model from epoch {checkpoint_q2['epoch'] + 1}")
    
    model_q2.eval()
    test_loss_q2 = 0.0
    test_dice_scores_q2 = []
    test_iou_scores_q2 = []
    test_acc_scores_q2 = []
    
    with torch.no_grad():
        for images, masks in test_loader_q2:
            images = images.to(device)
            masks = masks.to(device)
            
            outputs = model_q2(images)
            loss = criterion_q2(outputs, masks)
            
            preds = torch.argmax(outputs, dim=1)
            
            dice_scores = calculate_dice_score(preds, masks, CONFIG_Q2['num_classes'])
            iou_scores = calculate_iou(preds, masks, CONFIG_Q2['num_classes'])
            acc = calculate_pixel_accuracy(preds, masks)
            
            test_loss_q2 += loss.item()
            test_dice_scores_q2.append(dice_scores.cpu().numpy())
            test_iou_scores_q2.append(iou_scores.cpu().numpy())
            test_acc_scores_q2.append(acc)
    
    test_loss_q2 /= len(test_loader_q2)
    avg_dice_q2 = np.mean([d.mean() for d in test_dice_scores_q2])
    avg_iou_q2 = np.mean([i.mean() for i in test_iou_scores_q2])
    avg_acc_q2 = np.mean(test_acc_scores_q2)
    
    per_class_dice_q2 = np.mean(test_dice_scores_q2, axis=0)
    per_class_iou_q2 = np.mean(test_iou_scores_q2, axis=0)
    
    print("\n" + "=" * 60)
    print("Q2 TEST SET EVALUATION")
    print("=" * 60)
    print(f"Test Loss: {test_loss_q2:.4f}")
    print(f"Average Dice Score: {avg_dice_q2:.4f}")
    print(f"Average IoU Score: {avg_iou_q2:.4f}")
    print(f"Pixel Accuracy: {avg_acc_q2:.4f}")
    print("\nPer-class metrics:")
    for i in range(CONFIG_Q2['num_classes']):
        print(f"  Class {i}: Dice={per_class_dice_q2[i]:.4f}, IoU={per_class_iou_q2[i]:.4f}")
    print("=" * 60)
else:
    print("No Q2 training history available")

## 1-2. U-Net Model Architecture

In [ ]:
class DoubleConv(nn.Module):
    """(convolution => [BN] => ReLU) * 2"""
    def __init__(self, in_channels, out_channels, mid_channels=None):
        super().__init__()
        if not mid_channels:
            mid_channels = out_channels
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.double_conv(x)


class Down(nn.Module):
    """Downscaling with maxpool then double conv"""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels)
        )

    def forward(self, x):
        return self.maxpool_conv(x)


class Up(nn.Module):
    """Upscaling then double conv"""
    def __init__(self, in_channels, out_channels, bilinear=True):
        super().__init__()

        # if bilinear, use the normal convolutions to reduce the number of channels
        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
            self.conv = DoubleConv(in_channels, out_channels, in_channels // 2)
        else:
            self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
            self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x1, x2):
        x1 = self.up(x1)
        # input is CHW
        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]

        x1 = F.pad(x1, [diffX // 2, diffX - diffX // 2,
                        diffY // 2, diffY - diffY // 2])
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)


class OutConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(OutConv, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=1)

    def forward(self, x):
        return self.conv(x)


class UNet(nn.Module):
    def __init__(self, n_channels, n_classes, bilinear=False):
        super(UNet, self).__init__()
        self.n_channels = n_channels
        self.n_classes = n_classes
        self.bilinear = bilinear

        self.inc = DoubleConv(n_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        factor = 2 if bilinear else 1
        self.down4 = Down(512, 1024 // factor)
        self.up1 = Up(1024, 512 // factor, bilinear)
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)
        self.outc = OutConv(64, n_classes)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        logits = self.outc(x)
        return logits


# Create model
model = UNet(n_channels=CONFIG['in_channels'], n_classes=CONFIG['num_classes'], bilinear=False)
model = model.to(device)

# Print model summary
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model parameters: {count_parameters(model):,}")
print(f"\nModel architecture:")
print(model)

## 1-3. Loss Functions and Metrics

In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super(DiceLoss, self).__init__()
        self.smooth = smooth

    def forward(self, inputs, targets):
        # Apply softmax to get probabilities
        inputs = F.softmax(inputs, dim=1)
        
        # One-hot encode targets
        num_classes = inputs.shape[1]
        targets_one_hot = F.one_hot(targets, num_classes).permute(0, 3, 1, 2).float()
        
        # Flatten tensors
        inputs_flat = inputs.view(inputs.size(0), inputs.size(1), -1)
        targets_flat = targets_one_hot.view(targets_one_hot.size(0), targets_one_hot.size(1), -1)
        
        # Calculate Dice coefficient for each class
        intersection = (inputs_flat * targets_flat).sum(dim=2)
        union = inputs_flat.sum(dim=2) + targets_flat.sum(dim=2)
        
        dice = (2. * intersection + self.smooth) / (union + self.smooth)
        dice_loss = 1 - dice.mean()
        
        return dice_loss


class CombinedLoss(nn.Module):
    """Combination of Dice Loss and Cross Entropy Loss"""
    def __init__(self, dice_weight=0.5, ce_weight=0.5, smooth=1.0):
        super(CombinedLoss, self).__init__()
        self.dice_weight = dice_weight
        self.ce_weight = ce_weight
        self.dice_loss = DiceLoss(smooth=smooth)
        self.ce_loss = nn.CrossEntropyLoss()

    def forward(self, inputs, targets):
        dice = self.dice_loss(inputs, targets)
        ce = self.ce_loss(inputs, targets)
        return self.dice_weight * dice + self.ce_weight * ce


# Loss function
criterion = CombinedLoss(dice_weight=0.5, ce_weight=0.5)

# Metrics functions
def calculate_dice_score(pred, target, num_classes, smooth=1.0):
    """Calculate Dice score for each class"""
    pred_one_hot = F.one_hot(pred, num_classes).permute(0, 3, 1, 2).float()
    target_one_hot = F.one_hot(target, num_classes).permute(0, 3, 1, 2).float()
    
    pred_flat = pred_one_hot.view(pred_one_hot.size(0), pred_one_hot.size(1), -1)
    target_flat = target_one_hot.view(target_one_hot.size(0), target_one_hot.size(1), -1)
    
    intersection = (pred_flat * target_flat).sum(dim=2)
    union = pred_flat.sum(dim=2) + target_flat.sum(dim=2)
    
    dice = (2. * intersection + smooth) / (union + smooth)
    return dice.mean(dim=0)  # Average over batch, return per-class scores


def calculate_iou(pred, target, num_classes, smooth=1.0):
    """Calculate IoU (Intersection over Union) for each class"""
    pred_one_hot = F.one_hot(pred, num_classes).permute(0, 3, 1, 2).float()
    target_one_hot = F.one_hot(target, num_classes).permute(0, 3, 1, 2).float()
    
    pred_flat = pred_one_hot.view(pred_one_hot.size(0), pred_one_hot.size(1), -1)
    target_flat = target_one_hot.view(target_one_hot.size(0), target_one_hot.size(1), -1)
    
    intersection = (pred_flat * target_flat).sum(dim=2)
    union = pred_flat.sum(dim=2) + target_flat.sum(dim=2) - intersection
    
    iou = (intersection + smooth) / (union + smooth)
    return iou.mean(dim=0)  # Average over batch, return per-class scores


def calculate_pixel_accuracy(pred, target):
    """Calculate pixel-wise accuracy"""
    correct = (pred == target).sum().item()
    total = target.numel()
    return correct / total

## 1-4. Training Loop

In [ ]:
# Optimizer
optimizer = optim.Adam(model.parameters(), lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'])

# Learning rate scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5, verbose=True
)

# Training history
history = {
    'train_loss': [],
    'val_loss': [],
    'train_dice': [],
    'val_dice': [],
    'train_iou': [],
    'val_iou': [],
    'train_acc': [],
    'val_acc': []
}

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    running_dice = 0.0
    running_iou = 0.0
    running_acc = 0.0
    num_batches = 0
    
    for batch_idx, (images, masks) in enumerate(loader):
        images = images.to(device)
        masks = masks.to(device)
        
        # Forward pass
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        # Calculate metrics
        preds = torch.argmax(outputs, dim=1)
        dice_scores = calculate_dice_score(preds, masks, CONFIG['num_classes'])
        iou_scores = calculate_iou(preds, masks, CONFIG['num_classes'])
        acc = calculate_pixel_accuracy(preds, masks)
        
        running_loss += loss.item()
        running_dice += dice_scores.mean().item()
        running_iou += iou_scores.mean().item()
        running_acc += acc
        num_batches += 1
        
        if (batch_idx + 1) % 10 == 0:
            print(f'  Batch {batch_idx + 1}/{len(loader)}, Loss: {loss.item():.4f}, '
                  f'Dice: {dice_scores.mean().item():.4f}, IoU: {iou_scores.mean().item():.4f}, '
                  f'Acc: {acc:.4f}')
    
    return (running_loss / num_batches, running_dice / num_batches, 
            running_iou / num_batches, running_acc / num_batches)


def validate_epoch(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    running_dice = 0.0
    running_iou = 0.0
    running_acc = 0.0
    num_batches = 0
    
    with torch.no_grad():
        for images, masks in loader:
            images = images.to(device)
            masks = masks.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, masks)
            
            preds = torch.argmax(outputs, dim=1)
            dice_scores = calculate_dice_score(preds, masks, CONFIG['num_classes'])
            iou_scores = calculate_iou(preds, masks, CONFIG['num_classes'])
            acc = calculate_pixel_accuracy(preds, masks)
            
            running_loss += loss.item()
            running_dice += dice_scores.mean().item()
            running_iou += iou_scores.mean().item()
            running_acc += acc
            num_batches += 1
    
    return (running_loss / num_batches, running_dice / num_batches, 
            running_iou / num_batches, running_acc / num_batches)

In [ ]:
# Training loop
if len(volume_files) > 0:
    best_val_dice = 0.0
    patience_counter = 0
    patience = 10
    
    print("Starting training...")
    print(f"Total epochs: {CONFIG['epochs']}")
    print("-" * 60)
    
    for epoch in range(CONFIG['epochs']):
        print(f"\nEpoch {epoch + 1}/{CONFIG['epochs']}")
        print(f"Learning rate: {optimizer.param_groups[0]['lr']:.6f}")
        
        # Train
        train_loss, train_dice, train_iou, train_acc = train_epoch(
            model, train_loader, criterion, optimizer, device
        )
        
        # Validate
        val_loss, val_dice, val_iou, val_acc = validate_epoch(
            model, val_loader, criterion, device
        )
        
        # Update learning rate
        scheduler.step(val_loss)
        
        # Save history
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_dice'].append(train_dice)
        history['val_dice'].append(val_dice)
        history['train_iou'].append(train_iou)
        history['val_iou'].append(val_iou)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        
        print(f"\nTrain - Loss: {train_loss:.4f}, Dice: {train_dice:.4f}, "
              f"IoU: {train_iou:.4f}, Acc: {train_acc:.4f}")
        print(f"Val   - Loss: {val_loss:.4f}, Dice: {val_dice:.4f}, "
              f"IoU: {val_iou:.4f}, Acc: {val_acc:.4f}")
        
        # Save best model
        if val_dice > best_val_dice:
            best_val_dice = val_dice
            patience_counter = 0
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_dice': val_dice,
                'history': history
            }, f"{CONFIG['save_dir']}/best_model.pth")
            print(f"✓ Saved best model (Dice: {best_val_dice:.4f})")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"\nEarly stopping at epoch {epoch + 1}")
                break
    
    print("\n" + "=" * 60)
    print("Training completed!")
    print(f"Best validation Dice score: {best_val_dice:.4f}")
else:
    print("Skipping training - please provide dataset paths")

## 1-5. Training Curves Visualization

In [ ]:
if len(history['train_loss']) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Loss curves
    axes[0, 0].plot(history['train_loss'], label='Train Loss', linewidth=2)
    axes[0, 0].plot(history['val_loss'], label='Val Loss', linewidth=2)
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].set_title('Training and Validation Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # Dice score curves
    axes[0, 1].plot(history['train_dice'], label='Train Dice', linewidth=2)
    axes[0, 1].plot(history['val_dice'], label='Val Dice', linewidth=2)
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Dice Score')
    axes[0, 1].set_title('Training and Validation Dice Score')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # IoU curves
    axes[1, 0].plot(history['train_iou'], label='Train IoU', linewidth=2)
    axes[1, 0].plot(history['val_iou'], label='Val IoU', linewidth=2)
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('IoU Score')
    axes[1, 0].set_title('Training and Validation IoU Score')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # Accuracy curves
    axes[1, 1].plot(history['train_acc'], label='Train Acc', linewidth=2)
    axes[1, 1].plot(history['val_acc'], label='Val Acc', linewidth=2)
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Pixel Accuracy')
    axes[1, 1].set_title('Training and Validation Pixel Accuracy')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f"{CONFIG['save_dir']}/training_curves.png", dpi=300, bbox_inches='tight')
    plt.show()
else:
    print("No training history available")

## 1-6. Test Set Evaluation

In [ ]:
if len(volume_files) > 0:
    # Load best model
    checkpoint = torch.load(f"{CONFIG['save_dir']}/best_model.pth")
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"Loaded best model from epoch {checkpoint['epoch'] + 1}")
    
    # Evaluate on test set
    model.eval()
    test_loss = 0.0
    test_dice_scores = []
    test_iou_scores = []
    test_acc_scores = []
    
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for images, masks in test_loader:
            images = images.to(device)
            masks = masks.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, masks)
            
            preds = torch.argmax(outputs, dim=1)
            
            # Calculate per-class metrics
            dice_scores = calculate_dice_score(preds, masks, CONFIG['num_classes'])
            iou_scores = calculate_iou(preds, masks, CONFIG['num_classes'])
            acc = calculate_pixel_accuracy(preds, masks)
            
            test_loss += loss.item()
            test_dice_scores.append(dice_scores.cpu().numpy())
            test_iou_scores.append(iou_scores.cpu().numpy())
            test_acc_scores.append(acc)
            
            all_preds.append(preds.cpu())
            all_targets.append(masks.cpu())
    
    # Average metrics
    test_loss /= len(test_loader)
    avg_dice = np.mean([d.mean() for d in test_dice_scores])
    avg_iou = np.mean([i.mean() for i in test_iou_scores])
    avg_acc = np.mean(test_acc_scores)
    
    # Per-class metrics
    per_class_dice = np.mean(test_dice_scores, axis=0)
    per_class_iou = np.mean(test_iou_scores, axis=0)
    
    print("\n" + "=" * 60)
    print("TEST SET EVALUATION")
    print("=" * 60)
    print(f"Test Loss: {test_loss:.4f}")
    print(f"Average Dice Score: {avg_dice:.4f}")
    print(f"Average IoU Score: {avg_iou:.4f}")
    print(f"Pixel Accuracy: {avg_acc:.4f}")
    print("\nPer-class metrics:")
    for i in range(CONFIG['num_classes']):
        print(f"  Class {i}: Dice={per_class_dice[i]:.4f}, IoU={per_class_iou[i]:.4f}")
    print("=" * 60)
else:
    print("Skipping evaluation - please provide dataset paths")

## 1-7. Visualization of Predictions

In [ ]:
if len(volume_files) > 0:
    # Visualize some test predictions
    model.eval()
    num_samples = min(6, len(test_loader))
    
    fig, axes = plt.subplots(num_samples, 3, figsize=(15, 5 * num_samples))
    if num_samples == 1:
        axes = axes.reshape(1, -1)
    
    with torch.no_grad():
        for idx, (images, masks) in enumerate(test_loader):
            if idx >= num_samples:
                break
            
            images = images.to(device)
            masks = masks.to(device)
            
            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)
            
            # Take first image from batch
            img = images[0, 0].cpu().numpy()
            mask = masks[0].cpu().numpy()
            pred = preds[0].cpu().numpy()
            
            # Plot
            axes[idx, 0].imshow(img, cmap='gray')
            axes[idx, 0].set_title('Input Image')
            axes[idx, 0].axis('off')
            
            axes[idx, 1].imshow(mask, cmap='jet')
            axes[idx, 1].set_title('Ground Truth')
            axes[idx, 1].axis('off')
            
            axes[idx, 2].imshow(pred, cmap='jet')
            axes[idx, 2].set_title('Prediction')
            axes[idx, 2].axis('off')
    
    plt.tight_layout()
    plt.savefig(f"{CONFIG['save_dir']}/predictions.png", dpi=300, bbox_inches='tight')
    plt.show()
else:
    print("Skipping visualization - please provide dataset paths")